# SAE fidelity diagnostic — Colab driver

Go/no-go on using the ESMC-6B SAE as semantic axes for microproteins.
**Runtime:** GPU, prefer L4/A100 (6B bf16 ≈ 12 GB; do **not** quantize).
Cache goes to Google Drive so a disconnect just means re-running the extract cell.

In [ ]:
# 1. Get the repo + deps
!git clone https://github.com/gescobedo0/PLM-dark-matter.git 2>/dev/null || (cd PLM-dark-matter && git pull)
%cd PLM-dark-matter/sae_fidelity
!pip install -q -r requirements.txt

In [ ]:
# 2. Persist cache to Drive
from google.colab import drive; drive.mount('/content/drive')
import os; os.environ['SAE_DATA_DIR'] = '/content/drive/MyDrive/sae_fidelity_data'
os.makedirs(os.environ['SAE_DATA_DIR'], exist_ok=True)

In [ ]:
# 3. HF login (biohub/ESMC-6B may be gated; accept its license on the model page)
from huggingface_hub import login; login()

## Smoke run first (100/group) — verify the whole chain before scaling
Verification order that matters: **Group C (background) must reconstruct with low FVU**
and the **zinc-finger set must fire a metal feature** before any microprotein number is
trusted. A failure there means the layer index / loader / SAE weights are wrong.

In [ ]:
!python 01_prepare_subsample.py --smoke 100 --marker-limit 40
!python 02_extract_activations.py --limit 400
!python 03_sae_forward.py
!python 04_diagnostics.py
!python 05_report.py

## Full run (≈5k/group) — only after the smoke run looks sane
Extraction is resumable; re-run cell 02 if the session drops.

In [ ]:
!python 01_prepare_subsample.py
!python 02_extract_activations.py
!python 03_sae_forward.py
!python 04_diagnostics.py
!python 05_report.py

In [ ]:
# 4. Show the verdict
import os
from IPython.display import Markdown, Image, display
rep = os.path.join(os.environ['SAE_DATA_DIR'], 'report')
display(Markdown(open(os.path.join(rep, 'report.md')).read()))
display(Image(os.path.join(rep, 'fvu_ecdf.png')))